In [1]:
# =========================================================
# SELECCIÓN DE PAÍSES PARA LA PARTE NARRATIVA
# Cuatro tipos de caso, cada uno con respaldo estadístico explícito:
#   A. Democracias con exclusión más alta de lo esperado
#   B. Regímenes poco democráticos con exclusión más baja de lo
#      esperado (ojo: posible homogeneización forzada, no inclusión real)
#   C. Convergencia: mejoraron por igual desde extremos opuestos
#      de trayectoria democrática
#   D. Casos espejo: mismo nivel de exclusión en 2023, llegaron
#      ahí por caminos opuestos (uno mejoró, otro empeoró)
# =========================================================

In [5]:
# =========================================================
# CELDA 1: Cargar librerías y preparar la base de datos
# ---------------------------------------------------------
# Objetivo:
#   - Importar las librerías necesarias para el análisis.
#   - Leer la base de datos de V-Dem.
#   - Filtrar únicamente el período 2000–2023.
#
# Además:
#   Se define un diccionario con nombres descriptivos para
#   las principales variables, facilitando la presentación
#   de resultados y evitando utilizar las abreviaciones
#   originales de V-Dem.
#
# Resultado:
#   La base de datos queda lista para identificar países
#   candidatos para la parte narrativa del artículo.
# =========================================================
import pandas as pd
import numpy as np
import itertools
import statsmodels.api as sm

df = pd.read_csv('/content/vdem_exclusion_subset.csv')
df21 = df[(df['year'] >= 2000) & (df['year'] <= 2023)].copy()

# Nombres completos de las variables (para mostrar en vez de la abreviación V-Dem)
NOMBRES_VARIABLES = {
    'v2xpe_exlsocgr': 'Exclusión política por grupo social (etnia/religión)',
    'v2xpe_exlgender': 'Exclusión política por género',
    'v2x_polyarchy': 'Índice de poliarquía (nivel de democracia electoral)',
    'v2x_libdem': 'Índice de democracia liberal',
    'e_gdppc': 'PIB per cápita',
}

In [6]:
# =========================================================
# CELDA 2: Construir un resumen descriptivo por país
# ---------------------------------------------------------
# Objetivo:
#   Calcular el promedio de democracia y de las dos
#   dimensiones principales de exclusión para cada país
#   durante el período 2000–2023.
#
# Este resumen permite:
#   - Conocer el universo de países disponibles.
#   - Obtener estadísticas descriptivas que servirán para
#     definir algunos criterios posteriores (por ejemplo,
#     el umbral de democracia).
#
# Resultado:
#   Se genera una tabla resumen con un registro por país.
# =========================================================

resumen = df21.groupby('country_name').agg(
    **{
        'Índice de poliarquía (promedio 2000-2023)': ('v2x_polyarchy', 'mean'),
        'Exclusión por grupo social (promedio 2000-2023)': ('v2xpe_exlsocgr', 'mean'),
        'Exclusión por género (promedio 2000-2023)': ('v2xpe_exlgender', 'mean'),
    }
).dropna()

print(f"Países disponibles: {len(resumen)}")
resumen.head()


# =========================================================
# CASO A y B: ANOMALÍAS RESPECTO A LO ESPERADO
# Se usa el RESIDUAL de una regresión (exclusión ~ democracia +
# log(PIB per cápita) + región), igual que en el Método 1. Un
# residual alto = excluye más de lo que su democracia/riqueza
# predicen. Un residual bajo = excluye menos de lo esperado.
# =========================================================

Países disponibles: 179


,Índice de poliarquía (promedio 2000-2023),Exclusión por grupo social (promedio 2000-2023),Exclusión por género (promedio 2000-2023)
country_name,,,
Afghanistan,0.277042,0.748000,0.848792
Albania,0.514792,0.240250,0.235750
Algeria,0.315083,0.325042,0.359750
Angola,0.248917,0.745333,0.487042
Argentina,0.786042,0.112083,0.094333


In [8]:
# =========================================================
# CELDA 3: Calcular residuales para detectar anomalías
# ---------------------------------------------------------
# Objetivo:
#   Estimar cuánto se desvía cada país del nivel de
#   exclusión esperado según su democracia, riqueza y
#   región geográfica.
#
# Metodología:
#   Se ajusta una regresión lineal donde la exclusión es
#   explicada por:
#
#       • Democracia electoral.
#       • Logaritmo del PIB per cápita.
#       • Región geográfica.
#
# El residual promedio por país permite identificar:
#
#   Residual positivo:
#       Mayor exclusión de la esperada.
#
#   Residual negativo:
#       Menor exclusión de la esperada.
#
# Resultado:
#   Se obtiene un ranking de anomalías para cada país.
# =========================================================

def calcular_residuales(data, variable_objetivo):
    sub = data.dropna(subset=[variable_objetivo, 'v2x_polyarchy', 'e_gdppc', 'e_regiongeo']).copy()
    sub['log_gdppc'] = np.log(sub['e_gdppc'])
    X = pd.get_dummies(sub[['v2x_polyarchy', 'log_gdppc', 'e_regiongeo']],
                        columns=['e_regiongeo'], drop_first=True)
    X = sm.add_constant(X.astype(float))
    y = sub[variable_objetivo]
    modelo = sm.OLS(y, X).fit()
    sub['residual'] = modelo.resid

    ranking = (sub.groupby('country_name')
               .agg(residual=('residual', 'mean'),
                    democracia_promedio=('v2x_polyarchy', 'mean'),
                    nivel_exclusion_promedio=(variable_objetivo, 'mean'))
               .sort_values('residual', ascending=False))
    return ranking

In [9]:
# =========================================================
# CELDA 4: Identificar candidatos para los Casos A y B
# ---------------------------------------------------------
# Objetivo:
#   Aplicar el cálculo de residuales tanto para exclusión
#   por grupo social como para exclusión por género.
#
# Se presentan:
#
#   Caso A:
#       Países que excluyen más de lo esperado.
#
#   Caso B:
#       Países que excluyen menos de lo esperado.
#
# Estos resultados constituyen únicamente una primera
# selección estadística de posibles estudios de caso.
# =========================================================

residual_social = calcular_residuales(df21, 'v2xpe_exlsocgr')
residual_genero = calcular_residuales(df21, 'v2xpe_exlgender')

print(f"\n=== {NOMBRES_VARIABLES['v2xpe_exlsocgr']} ===")
print("\nCASO A — Excluyen MÁS de lo esperado (candidatos: revisar si son democracias):")
print(residual_social.head(10).round(3))
print("\nCASO B — Excluyen MENOS de lo esperado (candidatos: revisar régimen político):")
print(residual_social.tail(10).round(3))

print(f"\n=== {NOMBRES_VARIABLES['v2xpe_exlgender']} ===")
print("\nCASO A — Excluyen MÁS de lo esperado:")
print(residual_genero.head(10).round(3))
print("\nCASO B — Excluyen MENOS de lo esperado:")
print(residual_genero.tail(10).round(3))


=== Exclusión política por grupo social (etnia/religión) ===

CASO A — Excluyen MÁS de lo esperado (candidatos: revisar si son democracias):
               residual  democracia_promedio  nivel_exclusion_promedio
country_name                                                          
Mauritania        0.449                0.381                     0.912
Sudan             0.334                0.199                     0.925
South Africa      0.295                0.760                     0.512
Guatemala         0.288                0.573                     0.810
Paraguay          0.287                0.591                     0.726
Peru              0.285                0.779                     0.610
Bahrain           0.279                0.171                     0.860
Tajikistan        0.252                0.214                     0.852
Iraq              0.246                0.338                     0.825
Burma/Myanmar     0.232                0.216                     0.817

CASO 

In [10]:
# =========================================================
# CELDA 5: Refinar los Casos A y B según el régimen político
# ---------------------------------------------------------
# Objetivo:
#   Reducir falsos positivos aplicando un criterio sencillo
#   de clasificación política.
#
# Criterios utilizados:
#
#   Caso A:
#       Solo se conservan países con democracia superior o
#       igual a la mediana de la muestra.
#
#   Caso B:
#       Solo se conservan países con democracia inferior a
#       dicha mediana.
#
# Resultado:
#   Se obtiene una lista más consistente de candidatos para
#   la discusión narrativa.
# =========================================================

mediana_dem = resumen['Índice de poliarquía (promedio 2000-2023)'].median()

def filtrar_caso_A_democracias(residual_df, umbral_democracia=mediana_dem):
    return residual_df[residual_df['democracia_promedio'] >= umbral_democracia].head(10)

def filtrar_caso_B_autoritarios(residual_df, umbral_democracia=mediana_dem):
    return residual_df[residual_df['democracia_promedio'] < umbral_democracia].tail(10)

print("\nCASO A filtrado (democracias con exclusión social más alta de lo esperado):")
print(filtrar_caso_A_democracias(residual_social).round(3))

print("\nCASO B filtrado (regímenes poco democráticos con exclusión social más baja de lo esperado):")
print(filtrar_caso_B_autoritarios(residual_social).round(3))


CASO A filtrado (democracias con exclusión social más alta de lo esperado):
                    residual  democracia_promedio  nivel_exclusion_promedio
country_name                                                               
South Africa           0.295                0.760                     0.512
Guatemala              0.288                0.573                     0.810
Paraguay               0.287                0.591                     0.726
Peru                   0.285                0.779                     0.610
Dominican Republic     0.215                0.624                     0.473
Indonesia              0.210                0.667                     0.509
Suriname               0.169                0.760                     0.491
India                  0.148                0.649                     0.515
Slovakia               0.134                0.839                     0.124
Liberia                0.128                0.536                     0.574

CASO B fil

In [11]:
# =========================================================
# CELDA 6: Calcular cambios entre 2000 y 2023
# ---------------------------------------------------------
# Objetivo:
#   Medir cómo evolucionó cada país durante el período de
#   estudio tanto en democracia como en exclusión.
#
# Para cada país se calculan:
#
#   - Nivel inicial (2000).
#   - Nivel final (2023).
#   - Cambio en democracia.
#   - Cambio en exclusión.
#
# Resultado:
#   Se construye la base necesaria para comparar trayectorias
#   históricas entre países.
# =========================================================

def tabla_cambios(data, variable_objetivo):
    piv_dem = data.pivot(index='country_name', columns='year', values='v2x_polyarchy')
    piv_var = data.pivot(index='country_name', columns='year', values=variable_objetivo)
    tabla = pd.DataFrame({
        'democracia_2000': piv_dem[2000], 'democracia_2023': piv_dem[2023],
        'exclusion_2000': piv_var[2000], 'exclusion_2023': piv_var[2023]
    }).dropna()
    tabla['cambio_democracia'] = tabla['democracia_2023'] - tabla['democracia_2000']
    tabla['cambio_exclusion'] = tabla['exclusion_2023'] - tabla['exclusion_2000']
    return tabla

cambios_social = tabla_cambios(df21, 'v2xpe_exlsocgr')

In [12]:
# =========================================================
# CELDA 7: Buscar casos de convergencia
# ---------------------------------------------------------
# Objetivo:
#   Identificar pares de países que terminaron con niveles
#   similares de exclusión después de seguir trayectorias
#   democráticas claramente distintas.
#
# Criterios:
#   - Ambos mejoran su nivel de exclusión.
#   - Uno aumenta su democracia mientras el otro disminuye
#     o sigue una trayectoria opuesta.
#   - Ambos presentan cambios democráticos suficientemente
#     grandes para ser sustantivos.
#
# Resultado:
#   Se generan candidatos para ilustrar procesos de
#   convergencia mediante estudios comparados.
# =========================================================

def buscar_convergencia(tabla, mejora_minima=-0.08, cambio_dem_minimo=0.05):
    """
    Busca pares de países que mejoraron su exclusión de forma parecida
    (cambio_exclusion muy negativo en ambos) pero cuyo cambio de
    democracia tuvo signo OPUESTO (uno se democratizó, el otro no).
    """
    candidatos = tabla[tabla['cambio_exclusion'] <= mejora_minima].copy()
    pares = []
    for a, b in itertools.combinations(candidatos.index, 2):
        da, db = candidatos.loc[a], candidatos.loc[b]
        if (np.sign(da['cambio_democracia']) != np.sign(db['cambio_democracia'])
                and abs(da['cambio_democracia']) > cambio_dem_minimo
                and abs(db['cambio_democracia']) > cambio_dem_minimo):
            gap_final = abs(da['exclusion_2023'] - db['exclusion_2023'])
            pares.append({
                'pais_A': a, 'pais_B': b,
                'cambio_democracia_A': round(da['cambio_democracia'], 3),
                'cambio_democracia_B': round(db['cambio_democracia'], 3),
                'nivel_2023_A': round(da['exclusion_2023'], 3),
                'nivel_2023_B': round(db['exclusion_2023'], 3),
                'diferencia_nivel_final': round(gap_final, 3)
            })
    return pd.DataFrame(pares).sort_values('diferencia_nivel_final')

pares_convergencia = buscar_convergencia(cambios_social)
print(f"\n=== CASO C — Convergencia en {NOMBRES_VARIABLES['v2xpe_exlsocgr']} ===")
print("(mejoraron por igual, viniendo de trayectorias democráticas opuestas)")
print(pares_convergencia.head(10).to_string(index=False))


=== CASO C — Convergencia en Exclusión política por grupo social (etnia/religión) ===
(mejoraron por igual, viniendo de trayectorias democráticas opuestas)
 pais_A     pais_B  cambio_democracia_A  cambio_democracia_B  nivel_2023_A  nivel_2023_B  diferencia_nivel_final
   Chad       Iraq               -0.131                0.226         0.744         0.752                   0.008
Bolivia      Nepal               -0.157                0.325         0.326         0.281                   0.045
Bolivia     Kosovo               -0.157                0.438         0.326         0.272                   0.054
Bolivia The Gambia               -0.157                0.380         0.326         0.263                   0.063
Bolivia     Jordan               -0.157                0.066         0.326         0.397                   0.071
Bolivia      Kenya               -0.157                0.196         0.326         0.402                   0.076
 Angola       Chad                0.174             

In [13]:
# =========================================================
# CELDA 8: Buscar casos espejo
# ---------------------------------------------------------
# Objetivo:
#   Encontrar pares de países que llegan prácticamente al
#   mismo nivel de exclusión en 2023, pero recorriendo
#   trayectorias opuestas.
#
# Criterios:
#   - Un país empeora.
#   - El otro mejora.
#   - Ambos terminan con niveles finales muy similares.
#
# Resultado:
#   Se identifican casos útiles para comparar cómo caminos
#   diferentes pueden conducir a resultados semejantes.
# =========================================================

def buscar_espejo(tabla, umbral_cambio=0.05, umbral_gap_final=0.05):
    empeoraron = tabla[tabla['cambio_exclusion'] >= umbral_cambio]
    mejoraron = tabla[tabla['cambio_exclusion'] <= -umbral_cambio]

    pares = []
    for a in empeoraron.index:
        for b in mejoraron.index:
            gap = abs(tabla.loc[a, 'exclusion_2023'] - tabla.loc[b, 'exclusion_2023'])
            if gap < umbral_gap_final:
                pares.append({
                    'pais_que_empeoro': a, 'pais_que_mejoro': b,
                    'cambio_empeoro': round(tabla.loc[a, 'cambio_exclusion'], 3),
                    'cambio_mejoro': round(tabla.loc[b, 'cambio_exclusion'], 3),
                    'nivel_2023_ambos': round(tabla.loc[a, 'exclusion_2023'], 3),
                    'diferencia_nivel_final': round(gap, 3)
                })
    return pd.DataFrame(pares).sort_values('diferencia_nivel_final')

pares_espejo = buscar_espejo(cambios_social)
print(f"\n=== CASO D — Casos espejo en {NOMBRES_VARIABLES['v2xpe_exlsocgr']} ===")
print("(mismo destino en 2023, trayectorias opuestas)")
print(pares_espejo.head(10).to_string(index=False))


=== CASO D — Casos espejo en Exclusión política por grupo social (etnia/religión) ===
(mismo destino en 2023, trayectorias opuestas)
pais_que_empeoro pais_que_mejoro  cambio_empeoro  cambio_mejoro  nivel_2023_ambos  diferencia_nivel_final
            Iran        Ethiopia           0.085         -0.098             0.788                   0.000
       Mauritius           Kenya           0.211         -0.255             0.402                   0.000
         Lebanon            Fiji           0.234         -0.098             0.586                   0.000
          France         Georgia           0.064         -0.145             0.158                   0.001
         Senegal        Zanzibar           0.068         -0.085             0.289                   0.001
          Uganda            Iraq           0.051         -0.091             0.751                   0.001
     Philippines        Malaysia           0.127         -0.167             0.455                   0.001
           India  

In [15]:
# =========================================================
# CELDA 9: Repetir el análisis para exclusión de género
# ---------------------------------------------------------
# Objetivo:
#   Aplicar exactamente la misma metodología utilizada para
#   la exclusión por grupo social, pero empleando la
#   exclusión política por género como variable de interés.
#
# Esto permite identificar casos comparables en ambas
# dimensiones de exclusión utilizando criterios idénticos.
# =========================================================

cambios_genero = tabla_cambios(df21, 'v2xpe_exlgender')

pares_convergencia_genero = buscar_convergencia(cambios_genero)
print(f"\n=== CASO C — Convergencia en {NOMBRES_VARIABLES['v2xpe_exlgender']} ===")
print(pares_convergencia_genero.head(10).to_string(index=False))

pares_espejo_genero = buscar_espejo(cambios_genero)
print(f"\n=== CASO D — Casos espejo en {NOMBRES_VARIABLES['v2xpe_exlgender']} ===")
print(pares_espejo_genero.head(10).to_string(index=False))


=== CASO C — Convergencia en Exclusión política por género ===
   pais_A       pais_B  cambio_democracia_A  cambio_democracia_B  nivel_2023_A  nivel_2023_B  diferencia_nivel_final
  Albania     Botswana                0.103               -0.138         0.182         0.188                   0.006
   Bhutan     Botswana                0.441               -0.138         0.199         0.188                   0.011
Indonesia        Kenya               -0.128                0.196         0.332         0.320                   0.012
  Bolivia        Kenya               -0.157                0.196         0.334         0.320                   0.014
 Botswana       Jordan               -0.138                0.066         0.188         0.206                   0.018
  Bolivia Sierra Leone               -0.157                0.280         0.334         0.358                   0.024
Indonesia Sierra Leone               -0.128                0.280         0.332         0.358                   0.026


In [16]:
# =========================================================
# CELDA 10: Exportar los candidatos seleccionados
# ---------------------------------------------------------
# Objetivo:
#   Guardar en archivos CSV todos los rankings y pares de
#   países identificados durante el proceso de selección.
#
# Archivos generados:
#   - Casos A y B para exclusión social.
#   - Casos A y B para exclusión de género.
#   - Casos C (convergencia).
#   - Casos D (casos espejo).
#
# Estos archivos servirán como insumo para escoger los
# estudios de caso que serán desarrollados en la parte
# narrativa del artículo.
# =========================================================

residual_social.to_csv('/content/caso_A_B_exclusion_social.csv')
residual_genero.to_csv('/content/caso_A_B_exclusion_genero.csv')
pares_convergencia.to_csv('/content/caso_C_convergencia_social.csv', index=False)
pares_espejo.to_csv('/content/caso_D_espejo_social.csv', index=False)
pares_convergencia_genero.to_csv('/content/caso_C_convergencia_genero.csv', index=False)
pares_espejo_genero.to_csv('/content/caso_D_espejo_genero.csv', index=False)
print("\nExportado: 6 archivos CSV con los candidatos de cada caso.")


Exportado: 6 archivos CSV con los candidatos de cada caso.


In [17]:
# =========================================================
# NOTA METODOLÓGICA
# ---------------------------------------------------------
# Este procedimiento NO selecciona automáticamente los casos
# que aparecerán en el artículo. Su propósito es generar una
# lista de candidatos respaldados por criterios estadísticos,
# reduciendo el riesgo de escoger ejemplos únicamente porque
# confirman una narrativa previa.
#
# Antes de elegir un caso definitivo conviene verificar:
#
#   1. Que exista suficiente evidencia documental,
#      periodística o académica para reconstruir su contexto.
#
#   2. Que la interpretación sea coherente con el régimen
#      político del país, especialmente en los casos donde la
#      exclusión parece artificialmente baja en regímenes
#      autoritarios.
#
#   3. Que la magnitud de las diferencias sea sustantiva y
#      no únicamente estadísticamente detectable.
#
# Los resultados deben entenderse como un punto de partida
# para el análisis cualitativo, no como evidencia definitiva
# sobre el comportamiento de cada país.
# =========================================================